# Cobbleverse en la nube — versión transparenteReconstrucción de [kmille36/Colab-Cloud-Gaming](https://github.com/kmille36/Colab-Cloud-Gaming)(archivado el 18/06/2026) adaptada a Minecraft en vez de Steam.**Por qué está reescrito y no se usa el original:** los dos ficheros ejecutables del repo(`ColabSteam` y `colab-moonweb`) son binarios ELF generados con **shc**, que cifra un scriptbash con RC4 dentro de un ELF. No se puede leer qué hacen. El notebook original te pidemontar tu Google Drive *antes* de ejecutarlos como root. Aquí todo el código está a la vista.---### Antes de empezar, dos cosas1. **Esto va contra el ToS de Colab.** El FAQ oficial prohíbe *"using a remote desktop or SSH"*   y avisa de que esas sesiones *"may be terminated at any time without warning"*.   **Usa una cuenta de Google desechable**, no la tuya. No montes tu Drive personal.2. **Ejecuta primero la celda 2 (Diagnóstico).** Tarda 30 segundos y te dice si tu runtime   concreto puede hacer esto, antes de que inviertas una hora. Si dice `NO VIABLE`,   las celdas siguientes no van a funcionar y el porqué está explicado en la salida.

## 1 · Mantener viva la sesión

In [ ]:
#@title Reproduce este audio para que Colab no te desconecte { display-mode: "form" }%%html<b>Dale al play y deja la pestaña visible.</b><br/><audio autoplay loop controls       src="https://github.com/anars/blank-audio/raw/master/10-minutes-of-silence.mp3"></audio>

## 2 · Diagnóstico — ejecuta esto primeroNo instala nada ni modifica el sistema. Mide el runtime que te ha tocado y responde a unasola pregunta: **¿puede salir el vídeo UDP de este contenedor hasta tu PC?**

In [ ]:
%%writefile diagnostico.py#!/usr/bin/env python3"""Diagnóstico de viabilidad de cloud gaming sobre Google Colab.No instala nada y no toca el sistema. Solo mide el runtime que te ha tocado ydice si Sunshine/Moonlight puede funcionar ahí, con el porqué.La pregunta que responde es una sola: ¿existe un camino para que el vídeo UDPsalga de este contenedor y llegue a tu PC?Uso:  python3 diagnostico.py"""import fcntlimport jsonimport osimport platformimport reimport shutilimport structimport subprocessimport sysimport timeimport urllib.request# ---------------------------------------------------------------- utilidadesVERDE, ROJO, AMBAR, GRIS, NEGRITA, FIN = (    "\033[92m", "\033[91m", "\033[93m", "\033[90m", "\033[1m", "\033[0m")OK, FALLO, AVISO = f"{VERDE}OK{FIN}", f"{ROJO}FALLO{FIN}", f"{AMBAR}AVISO{FIN}"resultados = {}def titulo(texto):    print(f"\n{NEGRITA}{texto}{FIN}")    print("─" * 66)def linea(etiqueta, estado, detalle=""):    print(f"  [{estado}] {etiqueta:<34} {GRIS}{detalle}{FIN}")def sh(cmd, timeout=20):    """Ejecuta un comando y devuelve (rc, stdout+stderr). Nunca lanza."""    try:        p = subprocess.run(            cmd, shell=True, capture_output=True, text=True, timeout=timeout        )        return p.returncode, (p.stdout + p.stderr).strip()    except Exception as e:                                    # noqa: BLE001        return 127, str(e)# ------------------------------------------------------------ 1. plataformadef check_plataforma():    titulo("1. Plataforma")    en_colab = "google.colab" in sys.modules or os.path.isdir("/content")    linea(        "Entorno Google Colab",        OK if en_colab else AVISO,        "detectado" if en_colab else "no parece Colab; el diagnóstico sigue igual",    )    resultados["colab"] = en_colab    linea("Kernel", OK, platform.release())    linea("Distribución", OK, _distro())    # El AppArmor de Colab es el que aplica las restricciones de red.    perfil = _leer("/proc/self/attr/current", "desconocido").strip("\x00").strip()    restrictivo = "datalabvm" in perfil or "docker" in perfil    linea(        "Perfil AppArmor",        AVISO if restrictivo else OK,        perfil or "ninguno",    )    resultados["apparmor"] = perfildef _distro():    txt = _leer("/etc/os-release", "")    m = re.search(r'PRETTY_NAME="([^"]+)"', txt)    return m.group(1) if m else "desconocida"def _leer(ruta, defecto=""):    try:        with open(ruta, "r", errors="replace") as f:            return f.read()    except Exception:                                         # noqa: BLE001        return defecto# -------------------------------------------------------------- 2. hardwaredef check_hardware():    titulo("2. Hardware asignado")    # --- GPU    rc, out = sh("nvidia-smi --query-gpu=name,memory.total,driver_version "                 "--format=csv,noheader")    tiene_gpu = rc == 0 and out and "not found" not in out.lower()    if tiene_gpu:        linea("GPU", OK, out.splitlines()[0])    else:        linea("GPU", FALLO, "sin GPU — Entorno de ejecución > Cambiar tipo > T4")    resultados["gpu"] = out if tiene_gpu else None    # --- NVENC: sin esto no hay codificación por hardware y no hay stream usable    if tiene_gpu:        rc, out = sh("nvidia-smi --query-gpu=name --format=csv,noheader")        nombre = out.splitlines()[0] if out else ""        # T4, L4, A10G, RTX... llevan NVENC. A100 y H100 NO llevan NVENC.        sin_nvenc = any(x in nombre.upper() for x in ("A100", "H100", "TPU"))        linea(            "NVENC (codificador de vídeo)",            FALLO if sin_nvenc else OK,            f"{nombre} — esta GPU NO tiene NVENC" if sin_nvenc            else f"{nombre} — debería tener NVENC",        )        resultados["nvenc"] = not sin_nvenc    # --- CPU: lo que de verdad limita a Minecraft con mods    nucleos = os.cpu_count() or 0    modelo = ""    m = re.search(r"model name\s*:\s*(.+)", _leer("/proc/cpuinfo"))    if m:        modelo = m.group(1).strip()    mhz = re.search(r"cpu MHz\s*:\s*([\d.]+)", _leer("/proc/cpuinfo"))    freq = f" @ {float(mhz.group(1)):.0f} MHz" if mhz else ""    estado_cpu = OK if nucleos >= 6 else (AVISO if nucleos >= 4 else FALLO)    linea(f"CPU ({nucleos} vCPU)", estado_cpu, modelo + freq)    if nucleos <= 2:        print(f"       {ROJO}↳ 2 vCPU tienen que repartirse entre el juego, el "              f"escritorio\n         virtual, Sunshine y el túnel. Minecraft con "              f"249 mods\n         va a dar tirones aunque el stream funcione.{FIN}")    resultados["vcpu"] = nucleos    # --- RAM    m = re.search(r"MemTotal:\s+(\d+) kB", _leer("/proc/meminfo"))    ram_gb = int(m.group(1)) / 1024 / 1024 if m else 0    linea("RAM", OK if ram_gb >= 12 else AVISO, f"{ram_gb:.1f} GB")    resultados["ram_gb"] = round(ram_gb, 1)    # --- Disco: Cobbleverse son ~239 MB de mrpack que se expanden bastante más    libre_gb = shutil.disk_usage("/content" if os.path.isdir("/content") else "/").free / 1024**3    linea("Disco libre", OK if libre_gb >= 25 else AVISO, f"{libre_gb:.0f} GB")    resultados["disco_gb"] = round(libre_gb, 1)# ----------------------------------------------- 3. el test que decide todo# Bit 12 del mapa de capabilities de Linux.CAP_NET_ADMIN = 12# <linux/if_tun.h>TUNSETIFF = 0x400454CAIFF_TUN = 0x0001IFF_NO_PI = 0x1000def check_red():    titulo("3. Red — el test que decide si esto es posible")    print(f"{GRIS}  Moonlight manda el vídeo por UDP. Para que un paquete UDP entre\n"          f"  en este contenedor hace falta una VPN (Tailscale/WireGuard), y toda\n"          f"  VPN necesita un dispositivo TUN. Eso es lo que se prueba aquí.{FIN}\n")    # --- 3.1 ¿Tenemos CAP_NET_ADMIN en el bounding set?    capbnd = 0    m = re.search(r"CapBnd:\s*([0-9a-fA-F]+)", _leer("/proc/self/status"))    if m:        capbnd = int(m.group(1), 16)    tiene_netadmin = bool(capbnd & (1 << CAP_NET_ADMIN))    linea(        "CAP_NET_ADMIN disponible",        OK if tiene_netadmin else FALLO,        f"CapBnd=0x{capbnd:016x}" + ("" if tiene_netadmin else " — capability ausente"),    )    resultados["cap_net_admin"] = tiene_netadmin    # --- 3.2 ¿Existe /dev/net/tun? Si no, ¿podemos crearlo?    existe_tun = os.path.exists("/dev/net/tun")    if not existe_tun:        os.makedirs("/dev/net", exist_ok=True)        rc, out = sh("mknod /dev/net/tun c 10 200 && chmod 600 /dev/net/tun")        existe_tun = os.path.exists("/dev/net/tun")        linea(            "/dev/net/tun",            OK if existe_tun else FALLO,            "creado con mknod" if existe_tun else f"mknod falló: {out[:40]}",        )    else:        linea("/dev/net/tun", OK, "ya existía")    resultados["dev_tun"] = existe_tun    # --- 3.3 La prueba real: abrir el dispositivo y crear la interfaz.    #     Esto es lo que hace tailscaled/wireguard por dentro. Si falla aquí,    #     falla para cualquier VPN, sin excepción.    tun_ok, motivo = False, "no se intentó"    if existe_tun:        try:            fd = os.open("/dev/net/tun", os.O_RDWR)            try:                ifr = struct.pack("16sH22s", b"diagtun0", IFF_TUN | IFF_NO_PI, b"")                fcntl.ioctl(fd, TUNSETIFF, ifr)                tun_ok, motivo = True, "interfaz creada correctamente"            finally:                os.close(fd)        except PermissionError as e:            motivo = f"operación no permitida ({e.errno}) — bloqueado por el sandbox"        except OSError as e:            motivo = f"errno {e.errno}: {e.strerror}"    linea("Crear interfaz TUN (ioctl real)", OK if tun_ok else FALLO, motivo)    resultados["tun_funciona"] = tun_ok    # --- 3.4 Salida a Internet (esto sí funciona siempre; es lo único que hay)    rc, _ = sh("timeout 8 curl -sf -o /dev/null https://api.github.com")    linea("Salida TCP a Internet", OK if rc == 0 else FALLO,          "sin restricción" if rc == 0 else "sin salida")    resultados["salida_tcp"] = rc == 0    # --- 3.5 ¿Hay IP pública / puertos entrantes? (spoiler: no)    linea("Puertos entrantes públicos", FALLO,          "Colab no expone puertos: no hay IP pública ni port-forward")    resultados["puertos_entrantes"] = False    # --- 3.6 Región, para estimar la latencia hasta ti    try:        with urllib.request.urlopen("https://ipinfo.io/json", timeout=8) as r:            info = json.load(r)        loc = f"{info.get('city','?')}, {info.get('region','?')} ({info.get('country','?')})"        linea("Región del runtime", OK, loc)        resultados["region"] = loc    except Exception:                                         # noqa: BLE001        linea("Región del runtime", AVISO, "no se pudo determinar")# --------------------------------------------------------------- veredictodef veredicto():    titulo("VEREDICTO")    tun = resultados.get("tun_funciona", False)    gpu = resultados.get("gpu") is not None    vcpu = resultados.get("vcpu", 0)    if not gpu:        print(f"{ROJO}{NEGRITA}  NO VIABLE — no hay GPU asignada.{FIN}")        print("  Cambia el tipo de entorno de ejecución a T4 y vuelve a ejecutar.")        return "sin_gpu"    if tun:        print(f"{VERDE}{NEGRITA}  CAMINO ABIERTO — se puede crear un TUN.{FIN}\n")        print("  Algo ha cambiado en Colab respecto a lo documentado. Se puede montar")        print("  Tailscale en modo normal y Moonlight tendría su ruta UDP directa.")        print(f"  {NEGRITA}Sigue con la celda 2 del notebook.{FIN}")        if vcpu <= 2:            print(f"\n  {AMBAR}Aviso: con {vcpu} vCPU el stream irá, pero Cobbleverse")            print(f"  dará tirones. El límite pasa a ser la CPU, no la red.{FIN}")        return "viable"    # Caso real esperado    print(f"{ROJO}{NEGRITA}  NO VIABLE para streaming de baja latencia.{FIN}\n")    print("  La cadena de restricciones, en orden:\n")    print(f"    1. El contenedor no tiene {NEGRITA}CAP_NET_ADMIN{FIN} "          f"({'ausente' if not resultados.get('cap_net_admin') else 'presente pero insuficiente'}).")    print("    2. Sin esa capability no se puede crear una interfaz TUN,")    print("       aunque /dev/net/tun exista.")    print("    3. Sin TUN no hay Tailscale ni WireGuard ni ninguna VPN.")    print("    4. Colab tampoco expone puertos entrantes.")    print("    5. Por tanto el único transporte posible es un túnel HTTP/TCP")    print("       saliente (Cloudflare Tunnel, ngrok).")    print(f"    6. Y el vídeo de Moonlight es {NEGRITA}UDP{FIN}. Meterlo por TCP")    print("       provoca head-of-line blocking: cada paquete perdido congela")    print("       la imagen hasta que se retransmite.\n")    print(f"  {GRIS}Eso es exactamente el issue #11 del repo original")    print(f"  ('High latency with Cloudflare tunnel') y el motivo de que")    print(f"  el proyecto se archivara el 18/06/2026.{FIN}\n")    print(f"  {NEGRITA}Alternativas que sí funcionan: ver docs/cloud-gaming.md{FIN}")    return "no_viable"# ------------------------------------------------------------------ maindef main():    print(f"\n{NEGRITA}Diagnóstico de cloud gaming — PokeReport{FIN}")    print(f"{GRIS}{time.strftime('%Y-%m-%d %H:%M:%S')} · no modifica el sistema{FIN}")    check_plataforma()    check_hardware()    check_red()    v = veredicto()    ruta = "/content/diagnostico-resultado.json" if os.path.isdir("/content") \        else "diagnostico-resultado.json"    resultados["veredicto"] = v    try:        with open(ruta, "w") as f:            json.dump(resultados, f, indent=2, ensure_ascii=False)        print(f"\n{GRIS}Resultado guardado en {ruta}{FIN}")    except Exception:                                         # noqa: BLE001        pass    return 0 if v == "viable" else 1if __name__ == "__main__":    sys.exit(main())

In [ ]:
!python3 diagnostico.py

### Cómo leer el resultado| Veredicto | Qué significa | Qué hacer ||---|---|---|| `CAMINO ABIERTO` | Se pudo crear una interfaz TUN. Colab ha cambiado respecto a lo documentado. | Sigue en la celda 3 || `NO VIABLE` | Sin `CAP_NET_ADMIN` no hay TUN → no hay VPN → no hay ruta UDP. | Ver `docs/cloud-gaming.md` || `sin GPU` | No te han asignado T4. | *Entorno de ejecución → Cambiar tipo → T4*, y repite |**Por qué `NO VIABLE` no tiene arreglo desde aquí:** Moonlight manda el vídeo por UDP.Colab no expone puertos entrantes, así que la única forma de recibir UDP sería una VPN,y toda VPN necesita un dispositivo TUN que el sandbox de Colab no deja crear. Lo único quequeda es un túnel HTTP/TCP saliente, y meter vídeo en tiempo real por TCP causa*head-of-line blocking*: cada paquete perdido congela la imagen hasta que se retransmite.Eso es literalmente el issue **#11 del repo original** (*"High latency with Cloudflare tunnel"*),y la razón de que se archivara.

---## 3 · Entorno gráfico con aceleración por GPUSolo si el diagnóstico dijo `CAMINO ABIERTO`.El montaje es: **Xvfb** da el display, **VirtualGL con backend EGL** hace que OpenGL serenderice en la T4 (sin necesidad de un servidor X con driver NVIDIA, que en un contenedorno se puede levantar), y **Sunshine** captura ese display y lo codifica con **NVENC**.Sin VirtualGL, Minecraft renderizaría por software (llvmpipe) sobre 2 vCPU: la T4 soloestaría codificando vídeo y el juego iría a pocos FPS.

In [ ]:
%%bashset -euo pipefailecho "▶ Paquetes base…"apt-get update -qq# Sin `|| true`: si esto falla hay que verlo, no seguir a ciegas.# libnvidia-encode no se instala aquí a propósito: lo aporta el driver que Colab# ya trae, y pinear una versión que no case con el driver rompe NVENC.DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \    xvfb x11-utils xauth openbox \    libegl1 libgl1 libglx-mesa0 mesa-utils \    pulseaudio pulseaudio-utilsecho "▶ VirtualGL (render en GPU vía EGL)…"VGL=3.1.1curl -fsSL -o /tmp/vgl.deb \  "https://github.com/VirtualGL/virtualgl/releases/download/${VGL}/virtualgl_${VGL}_amd64.deb"DEBIAN_FRONTEND=noninteractive apt-get install -y -qq /tmp/vgl.debecho "▶ Sunshine…"SUN=$(curl -fsSL https://api.github.com/repos/LizardByte/Sunshine/releases/latest \      | grep -o 'https://[^"]*ubuntu-22.04-amd64.deb' | head -1)curl -fsSL -o /tmp/sunshine.deb "$SUN"DEBIAN_FRONTEND=noninteractive apt-get install -y -qq /tmp/sunshine.debecho "▶ Arrancando display virtual…"export DISPLAY=:0Xvfb :0 -screen 0 1920x1080x24 +extension GLX +extension RANDR &sleep 3openbox --sm-disable &pulseaudio --start --exit-idle-time=-1 2>/dev/null || trueechoecho "▶ Comprobación: ¿quién está renderizando OpenGL?"DISPLAY=:0 vglrun -d egl glxinfo 2>/dev/null | grep -E "OpenGL renderer" \  || echo "  ⚠ VirtualGL no pudo usar EGL — el render caería en llvmpipe (CPU)"

Si la última línea dice **`OpenGL renderer: ... Tesla T4 ...`**, el render va por GPU.Si dice **`llvmpipe`**, está renderizando por CPU y Cobbleverse será injugableindependientemente del stream.

---## 4 · Minecraft + Cobbleverse 1.7.42Tu launcher (`PokeReport Launcher`) **no sirve aquí**: `launcher/electron-builder.yml` solocompila target `nsis` para Windows x64 y esto es Linux. Se usa **Prism Launcher**, que es la*Opción B* de tu propio `docs/cliente.md` y acepta el mismo `.mrpack`.**Importante:** tiene que ser el `.mrpack` **exacto 1.7.42**, o el servidor te rechaza.Sube `client-pack/COBBLEVERSE 1.7.42.mrpack` a la raíz de tu Drive (el de la cuentadesechable) y ejecuta la celda.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')

In [ ]:
%%bashset -euo pipefailexport DISPLAY=:0MRPACK="/content/drive/MyDrive/COBBLEVERSE 1.7.42.mrpack"if [ ! -f "$MRPACK" ]; then  echo "✖ No encuentro el modpack en la raíz de tu Drive."  echo "  Sube: client-pack/COBBLEVERSE 1.7.42.mrpack"  exit 1fiecho "▶ Java 21…"DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-21-jreecho "▶ Prism Launcher…"curl -fsSL -o /tmp/prism.AppImage \  "$(curl -fsSL https://api.github.com/repos/PrismLauncher/PrismLauncher/releases/latest \     | grep -o 'https://[^\"]*x86_64.AppImage' | head -1)"chmod +x /tmp/prism.AppImage/tmp/prism.AppImage --appimage-extract >/dev/null 2>&1mv squashfs-root /opt/prismecho "▶ Importando el modpack (tarda: son 249 mods)…"/opt/prism/AppRun --import "$MRPACK" || trueechoecho "✔ Listo. Ajustes según tu docs/cliente.md:"echo "    · 6 GB de RAM al juego (no más)"echo "    · distancia de renderizado 8"echo "    · SIN shaders — Complementary duplica el coste de encoder también"

---## 5 · TúnelEsta celda **falla a propósito** si el diagnóstico dijo `NO VIABLE`, en vez de dejarte montarun túnel TCP que da 300 ms y parece que funciona hasta que intentas jugar.

In [ ]:
import json, subprocess, systry:    r = json.load(open("/content/diagnostico-resultado.json"))except FileNotFoundError:    sys.exit("✖ Ejecuta primero la celda 2 (Diagnóstico).")if not r.get("tun_funciona"):    print("✖ El diagnóstico dijo NO VIABLE: no se puede crear un TUN en este runtime.")    print()    print("  Montar aquí un túnel Cloudflare/ngrok te daría un stream por TCP, que es")    print("  exactamente lo que hacía el repo original y por lo que se archivó.")    print("  No merece la pena. Alternativas reales en docs/cloud-gaming.md")    sys.exit(1)print("✔ TUN disponible — instalando Tailscale…")subprocess.run("curl -fsSL https://tailscale.com/install.sh | sh", shell=True, check=True)subprocess.run("tailscaled --state=/var/lib/tailscale/tailscaled.state &",               shell=True, check=True)print()print("Ahora ejecuta en una celda nueva:  !tailscale up")print("Te dará una URL para autenticar. Después:  !tailscale ip -4")

---## 6 · Emparejar Moonlight1. `!tailscale ip -4` → te da la IP del runtime (ej. `100.x.y.z`).2. Instala Tailscale y **Moonlight** en tu PC, con la misma cuenta.3. En Moonlight, *Add PC* → esa IP. Te enseña un **PIN de 4 dígitos**.4. Ejecuta la celda de abajo con ese PIN.Es el mismo flujo que `moon-pair.sh` del repo original, que hacía dos llamadas a la API deSunshine: una para fijar las credenciales y otra para aceptar el PIN. Aquí la contraseña nose queda en `admin:admin`.

In [ ]:
PIN = ""  #@param {type:"string"}CLAVE_SUNSHINE = ""  #@param {type:"string"}import subprocess, sysif not PIN or not CLAVE_SUNSHINE:    sys.exit("Rellena el PIN de Moonlight y una contraseña para Sunshine.")subprocess.run([    "curl", "-u", "admin:admin", "-X", "POST", "-k",    "https://localhost:47990/api/password",    "-H", "Content-Type: application/json",    "-d", ('{"currentUsername":"admin","currentPassword":"admin",'           f'"newUsername":"admin","newPassword":"{CLAVE_SUNSHINE}",'           f'"confirmNewPassword":"{CLAVE_SUNSHINE}"}}'),], check=False)subprocess.run([    "curl", "-u", f"admin:{CLAVE_SUNSHINE}", "-X", "POST", "-k",    "https://localhost:47990/api/pin",    "-H", "Content-Type: application/json",    "-d", f'{{"pin":"{PIN}","name":"moonlight"}}',], check=False)print("\n✔ Emparejado. Lanza el juego desde Moonlight.")

---## 7 · PersistenciaColab **borra el disco entero** al cerrar la sesión. Sin copia, cada vez repites la descargade los 249 mods y la compilación de shaders.Esta celda comprime la instancia de Prism a tu Drive. Restaurar tarda bastante menos quereinstalar, pero sigue siendo varios minutos por sesión: es una limitación de la plataforma,no algo que se pueda optimizar.

In [ ]:
#@title Copia / restauración { display-mode: "form" }ACCION = "backup"  #@param ["backup", "restaurar"]import subprocess, osDESTINO = "/content/drive/MyDrive/cobbleverse-instancia.tar.zst"ORIGEN  = os.path.expanduser("~/.local/share/PrismLauncher")if ACCION == "backup":    subprocess.run(f'tar --zstd -cf "{DESTINO}" -C "{os.path.dirname(ORIGEN)}" '                   f'"{os.path.basename(ORIGEN)}"', shell=True, check=True)    print(f"✔ Guardado en {DESTINO}")else:    if not os.path.exists(DESTINO):        print("✖ No hay copia previa en el Drive.")    else:        os.makedirs(os.path.dirname(ORIGEN), exist_ok=True)        subprocess.run(f'tar --zstd -xf "{DESTINO}" -C "{os.path.dirname(ORIGEN)}"',                       shell=True, check=True)        print("✔ Restaurado.")

---## Límites que no desaparecen aunque todo lo anterior funcione| Límite | Valor ||---|---|| Tiempo de juego | ~4 h, y luego espera de 24 h || CPU | 2 vCPU compartidos — es el cuello de botella de Minecraft con mods || Persistencia | Ninguna; solo copias manuales a Drive || Riesgo | Suspensión de la cuenta de Google usada |Comparativa completa con las opciones de pago en [`docs/cloud-gaming.md`](../docs/cloud-gaming.md).